# Ifty YouTube Sentiment - Gen Z Model Training 🚀🧠
### GPU-Accelerated DistilBERT Fine-Tuning on Google Colab

**By Usman Ifty** | [GitHub](https://github.com/Usman-Ifty/yt-sentiment-mlops) | [LinkedIn](https://www.linkedin.com/in/usman-awan-a85877359/)

---
**Runtime → Change runtime type → T4 GPU** before running!

## Cell 1: Install Dependencies

In [ ]:
!pip install transformers datasets scikit-learn mlflow pandas numpy -q
print('✅ All packages installed!')

## Cell 2: Check GPU

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ No GPU found! Go to Runtime → Change runtime type → T4 GPU')

## Cell 3: Download & Build Master Dataset (80,000+ Gen Z records)

In [ ]:
import os, re, json, numpy as np, pandas as pd
from datasets import load_dataset

os.makedirs('data/raw', exist_ok=True)
frames = []

# --- Dataset 1: Cardiff NLP Tweet Sentiment (English, 3-class, high quality) ---
print('📥 [1/4] Cardiff NLP Tweet Sentiment...')
try:
    ds = load_dataset('cardiffnlp/tweet_sentiment_multilingual', 'english')
    for split in ['train', 'validation', 'test']:
        df = pd.DataFrame(ds[split]).rename(columns={'text': 'comment', 'label': 'sentiment'})
        df['sentiment'] = df['sentiment'].map({0: 'negative', 1: 'neutral', 2: 'positive'})
        frames.append(df[['comment', 'sentiment']])
    print(f'   ✅ Loaded {sum(len(pd.DataFrame(ds[s])) for s in ["train","validation","test"])} rows')
except Exception as e:
    print(f'   ⚠️ {e}')

# --- Dataset 2: MTEB Tweet Sentiment ---
print('📥 [2/4] MTEB Tweet Sentiment...')
try:
    ds2 = load_dataset('mteb/tweet_sentiment_extraction')
    for split in ds2.keys():
        df = pd.DataFrame(ds2[split])
        if 'text' in df.columns and 'label_text' in df.columns:
            df = df.rename(columns={'text': 'comment', 'label_text': 'sentiment'})
            frames.append(df[['comment', 'sentiment']])
    print(f'   ✅ Loaded')
except Exception as e:
    print(f'   ⚠️ {e}')

# --- Dataset 3: Social Media Multiclass Sentiment ---
print('📥 [3/4] Multiclass Social Sentiment...')
try:
    ds3 = load_dataset('Sp1786/multiclass-sentiment-analysis-dataset')
    for split in ds3.keys():
        df = pd.DataFrame(ds3[split])
        if 'text' in df.columns and 'label' in df.columns:
            df = df.rename(columns={'text': 'comment', 'label': 'sentiment'})
            df['sentiment'] = df['sentiment'].map({0: 'negative', 1: 'neutral', 2: 'positive'})
            frames.append(df[['comment', 'sentiment']])
    print(f'   ✅ Loaded')
except Exception as e:
    print(f'   ⚠️ {e}')

# --- Dataset 4: Financial + App Reviews (broader vocabulary) ---
print('📥 [4/4] Extra diverse sentiment sources...')
try:
    ds4 = load_dataset('tyqiangz/multilingual-sentiments', 'english')
    for split in ds4.keys():
        df = pd.DataFrame(ds4[split])
        if 'text' in df.columns and 'label' in df.columns:
            df = df.rename(columns={'text': 'comment', 'label': 'sentiment'})
            df['sentiment'] = df['sentiment'].map({0: 'negative', 1: 'neutral', 2: 'positive'})
            frames.append(df[['comment', 'sentiment']])
    print(f'   ✅ Loaded')
except Exception as e:
    print(f'   ⚠️ {e}')

# --- Merge, clean, deduplicate ---
master = pd.concat(frames, ignore_index=True).dropna(subset=['comment','sentiment'])
master['comment'] = master['comment'].astype(str).str.strip()
master['sentiment'] = master['sentiment'].str.lower().str.strip()
master = master[master['sentiment'].isin(['negative','neutral','positive'])]
master = master[master['comment'].str.len() > 3]
master = master.drop_duplicates(subset=['comment']).reset_index(drop=True)
master.to_csv('data/raw/master_genz_sentiment.csv', index=False)

print(f'\n{"="*50}')
print(f'MASTER DATASET BUILT: {len(master):,} records')
for lbl, cnt in master['sentiment'].value_counts().items():
    print(f'  {lbl:10s}: {cnt:,} ({cnt/len(master)*100:.1f}%)')
print(f'{"="*50}')

## Cell 4: Gen Z Slang Normalization & Tokenization

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset, DatasetDict
from transformers import DistilBertTokenizer

SLANG_MAP = {
    r'\bW\b': 'great win', r'\bL\b': 'bad loss',
    r'\bgoated\b': 'greatest of all time', r'\bgoat\b': 'greatest of all time',
    r'\bfire\b': 'amazing excellent', r'\blit\b': 'excellent exciting',
    r'\bbussin\b': 'delicious amazing', r'\bslay\b': 'impressive excellent',
    r'\bmid\b': 'average boring mediocre', r'\bfr\b': 'for real seriously',
    r'\bno cap\b': 'honestly seriously', r'\bcap\b': 'lie fake',
    r'\bngl\b': 'honestly', r'\bnpc\b': 'boring person no personality',
    r'\bdead\b': 'laughing so hard', r'\bbruh\b': 'wow unbelievable',
    r'\bsus\b': 'suspicious untrustworthy', r'\bcringe\b': 'embarrassing awkward',
    r'\bvibes\b': 'feeling energy', r'\btoxic\b': 'harmful negative bad',
    r'\bhype\b': 'excited enthusiastic', r'\brizz\b': 'charm charisma',
    r'\bno rizz\b': 'unattractive awkward', r'\bglow up\b': 'improvement transformation',
    r'\bcooked\b': 'in trouble done for', r'\bcleared\b': 'dominated excelled',
    r'\bhits different\b': 'feels unique special', r'\brent free\b': 'can not stop thinking',
    r'\bits giving\b': 'reminds me of feels like', r'\bbanger\b': 'excellent great song',
    r'\bpressed\b': 'upset bothered', r'\bstan\b': 'devoted fan supporter',
    r'\bclout\b': 'fame social influence', r'\bhating\b': 'criticizing jealously',
    r'💀': 'laughing so hard very funny', r'🔥': 'amazing excellent fire',
    r'🙌': 'great celebrating', r'👑': 'king greatest royalty',
    r'😂': 'very funny hilarious', r'😭': 'crying sad funny',
    r'💯': 'totally agree perfect', r'👎': 'bad dislike', r'👍': 'good agree',
    r'🤯': 'shocked mind blown', r'🥶': 'cold impressive',
}

def normalize_slang(text):
    for pattern, replacement in SLANG_MAP.items():
        text = re.sub(pattern, replacement, str(text), flags=re.IGNORECASE)
    return text.strip()

LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL = {0: 'negative', 1: 'neutral', 2: 'positive'}

df = pd.read_csv('data/raw/master_genz_sentiment.csv')
print(f'Normalizing {len(df):,} comments...')
df['comment_clean'] = df['comment'].apply(normalize_slang)
df['label'] = df['sentiment'].map(LABEL2ID)
df = df.dropna(subset=['label']).astype({'label': int})

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

os.makedirs('data/processed', exist_ok=True)
weights = compute_class_weight('balanced', classes=np.array([0,1,2]), y=train_df['label'].values)
with open('data/processed/class_weights.json', 'w') as f:
    json.dump({'weights': weights.tolist()}, f)
with open('data/processed/label_map.json', 'w') as f:
    json.dump({'id2label': {str(k): v for k,v in ID2LABEL.items()}, 'label2id': {v: str(k) for k,v in ID2LABEL.items()}}, f)

print('Tokenizing...')
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize_df(data_df):
    enc = tokenizer(data_df['comment_clean'].tolist(), truncation=True, max_length=128, padding='max_length')
    return Dataset.from_dict({'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask'], 'labels': data_df['label'].tolist()})

tokenized = DatasetDict({'train': tokenize_df(train_df), 'val': tokenize_df(val_df), 'test': tokenize_df(test_df)})
tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
tokenized.save_to_disk('data/processed/tokenized')
print('✅ Tokenization complete! Saved to data/processed/tokenized')

## Cell 5: Train on GPU with MLflow Tracking

In [ ]:
import torch, mlflow, mlflow.pytorch
from torch import nn
from torch.utils.data import DataLoader
from transformers import DistilBertForSequenceClassification, get_linear_schedule_with_warmup
from datasets import load_from_disk
from sklearn.metrics import accuracy_score, f1_score, classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {DEVICE}')

CONFIG = {
    'model_checkpoint': 'distilbert-base-uncased',
    'max_len': 128,
    'batch_size': 64,      # bigger batch = faster on GPU
    'epochs': 5,           # 5 epochs for a strong Gen Z model
    'learning_rate': 2e-5,
    'warmup_ratio': 0.1,
    'weight_decay': 0.01,
    'num_labels': 3,
}

# Load data
dataset = load_from_disk('data/processed/tokenized')
dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

train_loader = DataLoader(dataset['train'], batch_size=CONFIG['batch_size'], shuffle=True)
val_loader   = DataLoader(dataset['val'],   batch_size=CONFIG['batch_size']*2, shuffle=False)
test_loader  = DataLoader(dataset['test'],  batch_size=CONFIG['batch_size']*2, shuffle=False)

# Model
model = DistilBertForSequenceClassification.from_pretrained(CONFIG['model_checkpoint'], num_labels=3)
model.to(DEVICE)

# Loss with class weights
with open('data/processed/class_weights.json') as f:
    cw = json.load(f)['weights']
loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float).to(DEVICE))

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
total_steps = len(train_loader) * CONFIG['epochs']
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * CONFIG['warmup_ratio']), total_steps)

def evaluate(model, loader):
    model.eval()
    total_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            labs = batch['labels'].to(DEVICE)
            logits = model(input_ids=ids, attention_mask=mask).logits
            total_loss += loss_fn(logits, labs).item()
            preds_all.extend(torch.argmax(logits, dim=1).cpu().numpy())
            labels_all.extend(labs.cpu().numpy())
    return total_loss/len(loader), accuracy_score(labels_all, preds_all), f1_score(labels_all, preds_all, average='weighted'), preds_all, labels_all

# MLflow tracking
mlflow.set_experiment('yt-sentiment-genz-v2')
os.makedirs('models/bert/best_model', exist_ok=True)

with mlflow.start_run() as run:
    mlflow.log_params(CONFIG)
    best_f1 = 0

    for epoch in range(CONFIG['epochs']):
        model.train()
        total_train_loss = 0
        for step, batch in enumerate(train_loader):
            ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            labs = batch['labels'].to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(input_ids=ids, attention_mask=mask).logits, labs)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_train_loss += loss.item()
            if step % 50 == 0:
                print(f'  Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {loss.item():.4f}')

        val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader)
        print(f'\nEpoch {epoch+1} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}')
        mlflow.log_metrics({'train_loss': total_train_loss/len(train_loader), 'val_loss': val_loss, 'val_accuracy': val_acc, 'val_f1': val_f1}, step=epoch)

        if val_f1 > best_f1:
            best_f1 = val_f1
            model.save_pretrained('models/bert/best_model')
            print(f'  ✅ New best model saved! F1={val_f1:.4f}')

    # Final test evaluation
    from transformers import DistilBertForSequenceClassification as DBC
    best_model = DBC.from_pretrained('models/bert/best_model').to(DEVICE)
    test_loss, test_acc, test_f1, preds, labs = evaluate(best_model, test_loader)
    mlflow.log_metrics({'test_accuracy': test_acc, 'test_f1': test_f1})

    print(f'\n{"="*50}')
    print(f'TRAINING COMPLETE!')
    print(f'Test Accuracy : {test_acc:.4f}')
    print(f'Test F1       : {test_f1:.4f}')
    print(classification_report(labs, preds, target_names=['negative','neutral','positive']))
    print(f'MLflow Run ID : {run.info.run_id}')
    print(f'{"="*50}')

with open('models/bert/metrics.json', 'w') as f:
    json.dump({'test_accuracy': round(test_acc, 4), 'test_f1': round(test_f1, 4), 'val_f1': round(best_f1, 4)}, f, indent=2)
print('\n📊 metrics.json saved!')

## Cell 6: Download the Trained Model Files (as a ZIP)

In [ ]:
import shutil
from google.colab import files

# Zip the entire best_model folder
shutil.make_archive('ifty_genz_model', 'zip', 'models/bert/best_model')

# Also zip the processed data config
shutil.make_archive('ifty_data_config', 'zip', 'data/processed', '.')

print('✅ Model zipped!')
print('✅ Data config zipped!')
print('Downloading...')

files.download('ifty_genz_model.zip')   # model.safetensors, config.json
files.download('ifty_data_config.zip')  # class_weights.json, label_map.json

print('\n🎉 DONE! Extract the zip and copy to your local project:')
print('  ifty_genz_model.zip → models/bert/best_model/')
print('  ifty_data_config.zip → copy class_weights.json + label_map.json to data/processed/')